# Binôme B — Amélioration du modèle autoencodeur

**Phase CRISP-DM : Modeling.** Ce notebook compare la baseline convolutive MSE à trois variantes supervisées uniquement par la reconstruction de patches normaux : MAE, MAE + perte perceptuelle VGG19 gelé, et perte structurelle SSIM. Les masques servent seulement à sélectionner des patches normaux et à évaluer/calibrer les cartes sur validation ; ils ne sont jamais la cible d'entraînement.

> Le test a déjà été consulté lors d'une itération antérieure ; résultats exploratoires ; aucune décision modèle/post-traitement/XAI ne doit en découler. Ce notebook ne charge aucun split test.

Le modèle garde la même architecture et le même split. Le choix du modèle se fait d'abord sur l'AuPRC pixel de validation, puis sur le F1 moyen par image en cas d'égalité pratique. Le seuil et le lissage sont recalibrés séparément pour chaque variante sur validation.

## 1. Protocole, métriques et reproductibilité

In [ ]:
from pathlib import Path
import json
import platform
import sys

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Racine du dépôt introuvable (pyproject.toml).")
sys.path.insert(0, str(ROOT / "src"))

from roadanomaly import data_pipeline as dp, metrics
from roadanomaly.autoencoder import (
    PerceptualAutoencoder,
    anomaly_scores,
    build_autoencoder,
    build_vgg19_feature_model,
    compile_reconstruction_loss,
    extract_normal_patches,
    reconstruction_error_maps,
)
from roadanomaly.gpu import setup_gpu

SEED = 1337
IMG_SIZE = (352, 640)
MIN_OBJECT_PX = 4
SMALL_OBJECT_PX = round(1000 * (352 / 720) ** 2)
FPR_LIMIT = 0.3
PATCH = 64
STRIDE = 32
MASK_MARGIN = 4
ENCODER_FILTERS = (32, 64, 128, 256)
CODE_CHANNELS = 2
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
MAX_EPOCHS = 150
PATIENCE = 8
SIGMA_CANDIDATES = (0, 2, 4, 8, 16)
THRESHOLD_QUANTILES = np.linspace(0.50, 0.995, 60)
VGG_PERCEPTUAL_WEIGHT = 0.1

keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
DEVICE_NAME = setup_gpu()
MODEL_DIR = ROOT / "models"
FIG_DIR = ROOT / "reports" / "figures"
MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__, "| device:", DEVICE_NAME, "| small-object limit:", SMALL_OBJECT_PX)

## 2. Chargement train/validation et patches normaux

In [ ]:
train_frames, x_train, y_train = dp.load_split("train", IMG_SIZE)
val_frames, x_val, y_val = dp.load_split("val", IMG_SIZE)
assert x_train.shape[1:] == (*IMG_SIZE, 3) and y_train.shape[1:] == (*IMG_SIZE, 1)
assert x_val.shape[1:] == (*IMG_SIZE, 3) and y_val.shape[1:] == (*IMG_SIZE, 1)
assert set(map(str, train_frames)).isdisjoint(map(str, val_frames))
assert x_train.dtype == x_val.dtype == np.float32
assert np.isfinite(x_train).all() and np.isfinite(x_val).all()
assert x_train.min() >= 0 and x_train.max() <= 1
assert set(np.unique(y_train)).issubset({0.0, 1.0})

p_train = extract_normal_patches(x_train, y_train, PATCH, STRIDE, MASK_MARGIN)
p_val = extract_normal_patches(x_val, y_val, PATCH, STRIDE, MASK_MARGIN)
assert p_train.shape[1:] == p_val.shape[1:] == (PATCH, PATCH, 3)
print("Frames:", len(train_frames), "train /", len(val_frames), "validation")
print("Normal patches:", p_train.shape, "train /", p_val.shape, "validation")

def augment_patch(patch):
    patch = tf.image.random_flip_left_right(patch)
    return patch, patch

train_ds = (
    tf.data.Dataset.from_tensor_slices(p_train)
    .shuffle(len(p_train), seed=SEED)
    .map(augment_patch, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = tf.data.Dataset.from_tensor_slices((p_val, p_val)).batch(BATCH_SIZE)
gt_val = y_val[..., 0].astype(bool)

## 3. Expériences A et B — MSE, MAE, VGG19 gelé et SSIM

In [ ]:
def fit_variant(name, loss_name, feature_model=None, perceptual_weight=0.0):
    """Entraîne une variante à seed constant et retourne son modèle, historique et époque optimale."""
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    reconstruction_model = build_autoencoder(ENCODER_FILTERS, CODE_CHANNELS)
    if feature_model is None:
        training_model = reconstruction_model
        compile_reconstruction_loss(training_model, loss_name, LEARNING_RATE)
    else:
        training_model = PerceptualAutoencoder(
            reconstruction_model,
            feature_model=feature_model,
            perceptual_weight=perceptual_weight,
        )
        training_model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE))
    history = training_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE, restore_best_weights=True
        )],
        verbose=2,
    )
    model_path = MODEL_DIR / f"B_autoencoder_{name}.keras"
    reconstruction_model.save(model_path)
    best_epoch = int(np.argmin(history.history["val_loss"])) + 1
    return reconstruction_model, history.history, best_epoch, float(min(history.history["val_loss"])), model_path

# Baseline MSE et ablation MAE : même architecture, mêmes patches, split et seed.
variants = {}
variants["mse"] = fit_variant("mse", "mse")
variants["mae"] = fit_variant("mae", "mae")

# Le backbone VGG19 ImageNet reste gelé ; seuls les poids du décodeur/encodeur sont entraînés.
RUN_VGG19_EXPERIMENT = True
if RUN_VGG19_EXPERIMENT:
    vgg_features = build_vgg19_feature_model(PATCH)
    assert not vgg_features.trainable
    variants["mae_vgg19"] = fit_variant(
        "mae_vgg19", "mae", feature_model=vgg_features,
        perceptual_weight=VGG_PERCEPTUAL_WEIGHT,
    )

# SSIM est une quatrième variante ; tf.image.ssim utilise max_val=1 pour les pixels normalisés.
variants["ssim"] = fit_variant("ssim", "ssim")
print("Variantes entraînées :", list(variants))

## 4. Calibration validation-only et tableau comparatif

In [ ]:
def evaluate_variant(model, history, best_epoch, best_val_loss, model_path):
    """Choisit sigma et seuil sur validation et retourne les métriques du protocole."""
    reconstructions, raw_errors = reconstruction_error_maps(model, x_val)
    sigma_candidates = []
    for sigma in SIGMA_CANDIDATES:
        scores = anomaly_scores(raw_errors, sigma)
        sigma_candidates.append((
            float(average_precision_score(gt_val.ravel(), scores.ravel())), sigma, scores
        ))
    _, sigma, scores = max(sigma_candidates, key=lambda row: row[0])
    thresholds = np.quantile(scores, THRESHOLD_QUANTILES)
    threshold_f1 = []
    for threshold in thresholds:
        predicted = np.stack([
            metrics.remove_small_components(score >= threshold, MIN_OBJECT_PX)
            for score in scores
        ])
        threshold_f1.append(
            metrics.evaluate_masks(list(gt_val), list(predicted))["per_image_mean"]["f1"]
        )
    threshold = float(thresholds[int(np.argmax(threshold_f1))])
    predicted = np.stack([
        metrics.remove_small_components(score >= threshold, MIN_OBJECT_PX)
        for score in scores
    ])
    pixel = metrics.evaluate_masks(list(gt_val), list(predicted))
    objects = [metrics.object_scores(g, p, MIN_OBJECT_PX) for g, p in zip(gt_val, predicted)]
    sizes = [
        metrics.object_recall_by_size(g, p, MIN_OBJECT_PX, SMALL_OBJECT_PX)
        for g, p in zip(gt_val, predicted)
    ]
    rows = []
    for frame, truth, score, pred, pixel_row in zip(
        val_frames, gt_val, scores, predicted, pixel["per_image"]
    ):
        rows.append({
            "frame": frame,
            "auprc": float(average_precision_score(truth.ravel(), score.ravel())),
            **pixel_row,
        })
    result = {
        "model": model,
        "history": history,
        "reconstruction": reconstructions,
        "raw_error": raw_errors,
        "score": scores,
        "mask": predicted,
        "sigma": int(sigma),
        "threshold": threshold,
        "threshold_candidates": thresholds.astype(float).tolist(),
        "threshold_f1": [float(value) for value in threshold_f1],
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "model_path": str(model_path),
        "auprc": float(average_precision_score(gt_val.ravel(), scores.ravel())),
        "auroc": float(roc_auc_score(gt_val.ravel(), scores.ravel())),
        "aupro": float(metrics.aupro(list(gt_val), list(scores), MIN_OBJECT_PX, fpr_limit=FPR_LIMIT)),
        "pixel": pixel,
        "objects": objects,
        "sizes": sizes,
        "per_image": rows,
    }
    return result

results = {
    name: evaluate_variant(model, history, epoch, val_loss, model_path)
    for name, (model, history, epoch, val_loss, model_path) in variants.items()
}
comparison_rows = []
for name, result in results.items():
    obj_total = sum(row["gt_objects"] for row in result["objects"])
    obj_detected = sum(row["detected"] for row in result["objects"])
    fp_blobs = sum(row["false_alarms"] for row in result["objects"])
    size_total = {key: sum(row[key] for row in result["sizes"]) for key in (
        "small_detected", "small_total", "large_detected", "large_total"
    )}
    comparison_rows.append({
        "variant": name,
        "AuPRC": result["auprc"],
        "AUROC": result["auroc"],
        "AUPRO@0.3": result["aupro"],
        "F1_mean_image": result["pixel"]["per_image_mean"]["f1"],
        "F1_aggregated": result["pixel"]["aggregated"]["f1"],
        "Precision_mean": result["pixel"]["per_image_mean"]["precision"],
        "Recall_mean": result["pixel"]["per_image_mean"]["recall"],
        "F1_mean": result["pixel"]["per_image_mean"]["f1"],
        "IoU_mean": result["pixel"]["per_image_mean"]["iou"],
        "Dice_mean": result["pixel"]["per_image_mean"]["dice"],
        "Precision_aggregated": result["pixel"]["aggregated"]["precision"],
        "Recall_aggregated": result["pixel"]["aggregated"]["recall"],
        "F1_aggregated": result["pixel"]["aggregated"]["f1"],
        "IoU_aggregated": result["pixel"]["aggregated"]["iou"],
        "Dice_aggregated": result["pixel"]["aggregated"]["dice"],
        "Object_recall": obj_detected / obj_total if obj_total else 0.0,
        "Small_detected/total": f"{size_total['small_detected']}/{size_total['small_total']}",
        "Large_detected/total": f"{size_total['large_detected']}/{size_total['large_total']}",
        "FP_blobs": fp_blobs,
        "best_epoch": result["best_epoch"],
        "best_val_loss": result["best_val_loss"],
        "sigma": result["sigma"],
        "threshold": result["threshold"],
    })
comparison = pd.DataFrame(comparison_rows).sort_values(
    ["AuPRC", "F1_mean_image"], ascending=False, kind="stable"
).reset_index(drop=True)
print(comparison.round(4).to_string(index=False))

## 5. Faux positifs — inspection des patches de validation

In [ ]:
# Les fenêtres sont classées par nombre de pixels faux positifs avec le seuil de chaque variante gagnante.
WINNER = str(comparison.iloc[0]["variant"])
winning = results[WINNER]
fp_windows = []
for image_index, (image, truth, pred, score) in enumerate(
    zip(x_val, gt_val, winning["mask"], winning["score"])
):
    for top in range(0, IMG_SIZE[0] - PATCH + 1, STRIDE):
        for left in range(0, IMG_SIZE[1] - PATCH + 1, STRIDE):
            fp_count = int(np.count_nonzero(pred[top:top+PATCH, left:left+PATCH] & ~truth[top:top+PATCH, left:left+PATCH]))
            if fp_count:
                fp_windows.append((fp_count, image_index, top, left))
fp_windows.sort(reverse=True)
selected_fp = fp_windows[:20]
assert selected_fp, "Aucun patch contenant de faux positif n'a été trouvé sur validation."
fig, axes = plt.subplots(4, 5, figsize=(16, 13), tight_layout=True)
for axis, (fp_count, image_index, top, left) in zip(axes.flat, selected_fp):
    patch = x_val[image_index, top:top+PATCH, left:left+PATCH]
    fp_patch = winning["mask"][image_index, top:top+PATCH, left:left+PATCH].astype(bool) & ~gt_val[image_index, top:top+PATCH, left:left+PATCH]
    axis.imshow(patch)
    axis.imshow(fp_patch, cmap="Reds", alpha=0.45)
    axis.set_title(f"{val_frames[image_index]}\nFP={fp_count}", fontsize=7)
    axis.axis("off")
for axis in axes.flat[len(selected_fp):]: axis.axis("off")
plt.savefig(FIG_DIR / "B_06_validation_false_positive_patches.png", dpi=120, bbox_inches="tight")
plt.show()
print("Inspection visuelle requise : noter les motifs observés dans la cellule suivante.")

### Observations qualitatives documentées (à vérifier sur la grille)

La baseline du notebook 02 décrit déjà plusieurs motifs FP :
1. Le feuillage et la litière de feuilles peuvent produire une erreur élevée, même sans anomalie dans ces pixels.
2. Les textures de façades et de bâtiments génèrent des blobs FP dans l'arrière-plan.
3. Les gradients du ciel et les variations de texture du fond sont aussi signalés comme sources de fausses alertes.
4. Les textures à fort contraste (par exemple les bandes/stries) attirent la carte d'erreur, même si elles ne constituent pas un obstacle.

Ces observations qualitatives proviennent de l'analyse enregistrée du baseline ; vérifier sur la grille de la variante gagnante qu'elles restent applicables, et noter toute différence avant de figer le rapport.

## 6. Décision et sauvegarde des artefacts

In [ ]:
# Classement fixé ci-dessus : AuPRC prioritaire, F1 moyen par image en départage.
best_name = str(comparison.iloc[0]["variant"])
best = results[best_name]
best_config = {
    "artifact_version": 1,
    "model_path": Path(best["model_path"]).resolve().relative_to(MODEL_DIR.resolve()).as_posix(),
    "image_size": list(IMG_SIZE),
    "preprocessing": {
        "image_size_height_width": list(IMG_SIZE),
        "image_channels": "RGB",
        "image_dtype_range": "float32 [0, 1]",
        "image_resize": "bilinear",
        "mask_resize": "nearest",
    },
    "architecture": {
        "name": "convolutional_autoencoder",
        "encoder_filters": list(ENCODER_FILTERS),
        "code_channels": CODE_CHANNELS,
        "code_activation": "linear",
        "output_activation": "sigmoid",
    },
    "training": {
        "seed": SEED,
        "loss": best_name,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "maximum_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "best_epoch": best["best_epoch"],
        "best_validation_loss": best["best_val_loss"],
        "patch_size": PATCH,
        "patch_stride": STRIDE,
        "mask_margin": MASK_MARGIN,
        "normal_patches_train": int(len(p_train)),
        "normal_patches_validation": int(len(p_val)),
        "vgg19_frozen": best_name == "mae_vgg19",
        "vgg_perceptual_weight": VGG_PERCEPTUAL_WEIGHT if best_name == "mae_vgg19" else 0.0,
    },
    "postprocessing": {
        "sigma": best["sigma"],
        "threshold": best["threshold"],
        "per_image_z_score": True,
        "min_component_px": MIN_OBJECT_PX,
        "threshold_objective": "validation per-image mean F1",
    },
    "validation": {
        "frames": val_frames,
        "auprc": best["auprc"],
        "auroc": best["auroc"],
        "aupro_at_fpr_0_3": best["aupro"],
        "pixel": best["pixel"],
        "per_image": best["per_image"],
        "objects": best["objects"],
        "sizes": best["sizes"],
    },
    "comparison": comparison.to_dict(orient="records"),
    "history": best["history"],
    "sigma_auprc_table": [
        {"sigma": int(sigma), "auprc": float(auprc)}
        for auprc, sigma, _ in sorted(
            [(float(average_precision_score(gt_val.ravel(), anomaly_scores(results[best_name]["raw_error"], s).ravel())), s, None) for s in SIGMA_CANDIDATES],
            key=lambda row: row[1],
        )
    ],
    "test_policy": "Test consulté lors d'une itération antérieure ; résultats exploratoires ; aucune décision modèle/post-traitement/XAI ne doit en découler.",
}
config_path = MODEL_DIR / "B_best_config.json"
config_path.write_text(json.dumps(best_config, indent=2), encoding="utf-8")
print(f"Gagnant validation : {best_name} | AuPRC={best['auprc']:.6f} | F1={best['pixel']['per_image_mean']['f1']:.6f}")
print("Modèle :", best["model_path"])
print("Configuration :", config_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), tight_layout=True)
axes[0].plot(best["history"]["loss"], label="train")
axes[0].plot(best["history"]["val_loss"], label="validation")
axes[0].axvline(best["best_epoch"] - 1, color="gray", linestyle="--", label="meilleure époque")
axes[0].set_title(f"Courbes — {best_name}"); axes[0].set_xlabel("Époque"); axes[0].set_ylabel("loss"); axes[0].legend()
sigma_rows = best_config["sigma_auprc_table"]
axes[1].plot([r["sigma"] for r in sigma_rows], [r["auprc"] for r in sigma_rows], marker="o")
axes[1].axvline(best["sigma"], color="red", linestyle="--", label=f"sigma={best['sigma']}")
axes[1].set_title("Choix du sigma sur validation"); axes[1].set_xlabel("sigma"); axes[1].set_ylabel("AuPRC"); axes[1].legend()
plt.savefig(FIG_DIR / "B_01_learning_curves.png", dpi=120, bbox_inches="tight")
plt.savefig(FIG_DIR / "B_02_sigma_and_threshold.png", dpi=120, bbox_inches="tight")
plt.show()